In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ==========================================================
# MODEL PATH
# ==========================================================

# CHECKPOINT = "../V4B/V4B_Final_Merged_Model"
CHECKPOINT = "H:\\Compatifi Model\\V5A_Final_Merged_Model"

# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

print("Loading merged model...")
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print("✅ Model Loaded Successfully")

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 4/4 [01:15<00:00, 18.90s/it]


✅ Model Loaded Successfully


In [2]:
# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V4A.

Extract ONLY stable long-term memories.

Keep:
- Long-term preferences
- Long-term goals
- Stable personality traits
- Persistent habits
- Ongoing projects
- Important relationship facts
- Persistent health conditions

Ignore:
- Temporary emotions
- Greetings
- One-time events
- Short-term plans
- Casual conversation

Return ONLY valid JSON.

Example:
{"memories":["..."]}

If nothing should be remembered:

{"memories":[]}
"""

In [3]:
# ==========================================================
# TEST CONVERSATIONS
# ==========================================================

test_conversations = [

{
    "name": "ULTIMATE V4A TEST - DIFFERENT SCENARIO",
    "domain": "relationship",
    "relationship": "Mentor",
    "conversation": """
Mentor: You look busy lately.

User: Yeah, yesterday I spent the whole evening watching football highlights.

Mentor: Nice break.

User: It was. Today I'm back to normal.

Mentor: Still working on your language skills?

User: Yes. I've been practicing German every morning for nearly three years.

Mentor: That's impressive.

User: My long-term dream is to work as an AI researcher in Germany.

Mentor: You're getting closer every year.

User: I solve machine learning problems for at least two hours every day.

Mentor: Still writing technical blogs?

User: Yes. Writing about artificial intelligence has become one of my favorite hobbies.

Mentor: I remember you mentioning reading.

User: Every night before sleeping I read research papers for about an hour.

Mentor: That explains your progress.

User: I've always been curious and enjoy understanding how complex systems work.

Mentor: Any exercise these days?

User: I cycle every morning before breakfast.

Mentor: Good habit.

User: My asthma is under control now, but I still use my inhaler regularly.

Mentor: Glad to hear that.

User: Tomorrow I'm going shopping.

Mentor: Nice.

User: I also volunteer every Sunday teaching programming to high school students.

Mentor: That's wonderful.

User: Building educational AI tools has been my main personal project for the last two years.

Mentor: Still drinking coffee?

User: Nope. I switched to green tea years ago and haven't looked back.

Mentor: That's a healthy change.

User: Last weekend I went hiking with friends.

Mentor: Sounds fun.

User: Anyway, I should finish today's work.

Mentor: Good luck!
"""
}

]
# ==========================================================
# RUN TEST
# ==========================================================

for test in test_conversations:

    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    user_prompt = f"""
Domain: {test['domain']}
Relationship: {test['relationship']}
Instruction: Extract long-term memories

Conversation:

{test['conversation']}
"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    print("\n")
    print("=" * 100)
    print("MODEL OUTPUT")
    print("=" * 100)
    print(response)
    print("\n")

# ==========================================================
# EXPECTED MEMORIES
# ==========================================================


The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ULTIMATE V4A TEST - DIFFERENT SCENARIO


MODEL OUTPUT
<think>

</think>

{"memories": ["Practices German every morning and has done so for nearly three years.", "Has a long-term dream of working as an AI researcher in Germany.", "Reads research papers about artificial intelligence for an hour every night before sleeping.", "Cycles every morning before breakfast.", "Has asthma and uses an inhaler regularly.", "Volunteers every Sunday teaching programming to high school students.", "Has been working on a personal project to build educational AI tools for the last two years.", "Drinks green tea exclusively instead of coffee."]}




In [ ]:
# ====================================================================================================
# ULTIMATE V4A TEST - DIFFERENT SCENARIO
# ====================================================================================================
# The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# ====================================================================================================
# MODEL OUTPUT
# ====================================================================================================
# <think>

# </think>

# {"memories": ["Practices German every morning for nearly three years", "Reads research papers for an hour every night before sleeping", "Cycles every morning before breakfast", "Has a long-term goal to work as an AI researcher in Germany", "Volunteers every Sunday teaching programming to high school students", "Switched to green tea years ago and stopped drinking coffee"]}]}

